<a href="https://colab.research.google.com/github/LoneWolf2026/Neural-Network-for-Nuclear-Binding-Energy-Prediction-/blob/main/Nuclear_Binding_Energy_NeuralNet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#Omit uncertainties for now. Add them later. Focus on getting the Neural net trained first
#1. Optimize Gradient Descent functions
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader
from torchsummary import summary
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [4]:
data_df_2020 = pd.read_csv('/content/drive/MyDrive/AME_Dataset_2020.csv',header =None)
data_df_2020 = data_df_2020.drop([0, 1]).reset_index(drop=True) #removes neutron and proton from first two rows
data_df_2020 = data_df_2020.drop(columns=[0,4]).reset_index(drop=True)
data_df_2020.columns = range(data_df_2020.columns.size)
data_df_2020.head()

,0,1,2,3,4,5,6,7,8
0,1,1,2,13135.722895,0.000015,1112.28310,0.00020,14101.777844,0.000015
1,2,1,3,14949.810900,0.000080,2827.26540,0.00030,16049.281320,0.000080
2,1,2,3,14931.218880,0.000060,2572.68044,0.00015,16029.321970,0.000060
3,0,3,3,28667.000000,2000.000000,-2267.00000,667.00000,30775.000000,2147.000000
4,3,1,4,24621.129000,100.000000,1720.44910,25.00000,26431.867000,107.354000


In [5]:
og_data = data_df_2020.copy()


In [6]:
input = np.array(data_df_2020.iloc[:,[0,1,2,3,7]].values) #Neutrons, Protons, Mass Number, Mass Excess, and Atomic Mass are take as inputs
output = np.array(data_df_2020.iloc[:,[5]].values) #Binding Energy is our output

print(input)
print(output)


[[1.00000000e+00 1.00000000e+00 2.00000000e+00 1.31357229e+04
  1.41017778e+04]
 [2.00000000e+00 1.00000000e+00 3.00000000e+00 1.49498109e+04
  1.60492813e+04]
 [1.00000000e+00 2.00000000e+00 3.00000000e+00 1.49312189e+04
  1.60293220e+04]
 ...
 [1.77000000e+02 1.17000000e+02 2.94000000e+02 1.96397000e+05
  2.10840000e+05]
 [1.76000000e+02 1.18000000e+02 2.94000000e+02 1.99320000e+05
  2.13979000e+05]
 [1.77000000e+02 1.18000000e+02 2.95000000e+02 2.01369000e+05
  2.16178000e+05]]
[[1112.2831 ]
 [2827.2654 ]
 [2572.68044]
 ...
 [7092.     ]
 [7079.     ]
 [7076.     ]]


In [20]:
input_train,input_test,output_train,output_test = train_test_split(input,output,test_size = 0.2)

#Standardize input and output training data for simpler training
scaler_input = StandardScaler()
scaler_output = StandardScaler()
#must have two different scaler functions since input and output have different column dimensions

input_train = scaler_input.fit_transform(input_train)
output_train = scaler_output.fit_transform(output_train)

input_test = scaler_input.transform(input_test)
output_test = scaler_output.transform(output_test)

In [21]:
class dataset(Dataset):
  def __init__(self,input,output):
    self.input = torch.tensor(input, dtype = torch.float32).to(device)
    self.output = torch.tensor(output, dtype = torch.float32).to(device)

  def __len__(self):
    return len(self.input)

  def __getitem__(self,index):
    return self.input[index], self.output[index]

In [24]:
training_data = dataset(input_train,output_train)
testing_data = dataset(input_test,output_test)

In [23]:
train_dataloader = DataLoader(training_data,batch_size = 8,shuffle = True)
test_dataloader = DataLoader(testing_data,batch_size = 8,shuffle = True)

In [25]:
for i,b in train_dataloader:
  print(i)
  print("===========")
  print(b)
  break

tensor([[ 0.8835,  1.2024,  1.0159,  0.7905, -1.8054],
        [ 0.0969,  0.1967,  0.1370, -0.9220,  0.5099],
        [-0.4814, -0.8449, -0.6284, -0.1225,  0.6377],
        [ 1.6237,  1.3101,  1.5121,  1.5441, -1.6850],
        [ 2.0633,  1.9925,  2.0507,  3.3036, -1.4037],
        [ 2.1095,  2.0284,  2.0933,  3.4312, -1.3833],
        [-0.3888, -0.1984, -0.3166, -1.0725,  0.4858],
        [ 0.3977,  0.6277,  0.4914, -0.3523,  0.6010]], device='cuda:0')
tensor([[-0.4639],
        [ 0.4076],
        [ 0.2099],
        [-0.6354],
        [-1.1011],
        [-1.1263],
        [ 0.6831],
        [ 0.0327]], device='cuda:0')


In [12]:

class BindingE_NeuralNet(nn.Module):
  def __init__(self):
    super(BindingE_NeuralNet,self).__init__()

    self.layer1 = nn.Linear(input_train.shape[1],64)
    self.layer2 = nn.Linear(64,32)
    self.OutLayer = nn.Linear(32,1) #Output is Binding Energy
    self.Relu = nn.ReLU()

  def forward(self,X):

    X = self.Relu(self.layer1(X))
    X = self.Relu(self.layer2(X))
    X = self.OutLayer(X)

    #Return to this Model and refine architecture

    return X


BE_NeuralNet = BindingE_NeuralNet().to('cuda')

In [13]:
summary(BE_NeuralNet,(input.shape[1],))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Linear-1                   [-1, 64]             384
              ReLU-2                   [-1, 64]               0
            Linear-3                   [-1, 32]           2,080
              ReLU-4                   [-1, 32]               0
            Linear-5                    [-1, 1]              33
Total params: 2,497
Trainable params: 2,497
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.01
Estimated Total Size (MB): 0.01
----------------------------------------------------------------


In [26]:
criterion = nn.MSELoss() #Mean Square Error Loss function
optimizer = Adam(BE_NeuralNet.parameters(), lr = 0.001)

In [27]:
for data in train_dataloader:
  inputs, labels = data
  print(inputs)
  print(labels)
  break

tensor([[ 1.7394,  1.7052,  1.7389,  2.2545, -1.5714],
        [-1.2910, -1.2399, -1.2805, -0.4268,  0.5891],
        [-1.2216, -0.8449, -1.0820,  0.1141,  0.6755],
        [-0.7821, -1.0963, -0.9119, -0.1243,  0.6374],
        [-1.6380, -1.4554, -1.5782,  0.8986, -1.7882],
        [ 0.9529,  0.7714,  0.8883, -0.0163,  0.6547],
        [-0.7358, -0.9526, -0.8269, -0.5949,  0.5622],
        [ 0.0507,  0.1608,  0.0945, -1.0012,  0.4972]], device='cuda:0')
tensor([[-0.8512],
        [ 1.0071],
        [-0.0623],
        [ 0.3286],
        [-1.9193],
        [-0.1058],
        [ 0.7284],
        [ 0.4638]], device='cuda:0')


In [84]:
#Testing cell

tolerance = 0.1
mae = []
for data in train_dataloader:
    inputs, labels = data

    prediction = BE_NeuralNet(inputs)

    batch_loss = criterion(prediction,labels)
    total_loss_train += batch_loss.item()

    abs_value = torch.abs(prediction - labels)
    acc_train = (abs_value < tolerance).float().mean()

    mae.append(torch.mean(abs_value)) #mean absolute error

print(max(mae))
print(min(mae))

[tensor(0.0150, device='cuda:0', grad_fn=<MeanBackward0>), tensor(0.0091, device='cuda:0', grad_fn=<MeanBackward0>), tensor(0.0063, device='cuda:0', grad_fn=<MeanBackward0>), tensor(0.0091, device='cuda:0', grad_fn=<MeanBackward0>), tensor(0.0465, device='cuda:0', grad_fn=<MeanBackward0>), tensor(0.0480, device='cuda:0', grad_fn=<MeanBackward0>), tensor(0.0643, device='cuda:0', grad_fn=<MeanBackward0>), tensor(0.0061, device='cuda:0', grad_fn=<MeanBackward0>), tensor(0.0047, device='cuda:0', grad_fn=<MeanBackward0>), tensor(0.0074, device='cuda:0', grad_fn=<MeanBackward0>), tensor(0.0036, device='cuda:0', grad_fn=<MeanBackward0>), tensor(0.0099, device='cuda:0', grad_fn=<MeanBackward0>), tensor(0.0279, device='cuda:0', grad_fn=<MeanBackward0>), tensor(0.0300, device='cuda:0', grad_fn=<MeanBackward0>), tensor(0.0053, device='cuda:0', grad_fn=<MeanBackward0>), tensor(0.0071, device='cuda:0', grad_fn=<MeanBackward0>), tensor(0.0200, device='cuda:0', grad_fn=<MeanBackward0>), tensor(0.0055

In [95]:
total_loss_train_plot = []
total_loss_test_plot = []
total_acc_train_plot = []
total_acc_test_plot = []

tolerance = 0.1 #error tolerance of 0.1 keV

epochs = 20
for epoch in range(epochs):
  total_acc_train = []
  total_loss_train = 0
  total_acc_test = []
  total_loss_test = 0

  for data in train_dataloader:
    inputs, labels = data

    prediction = BE_NeuralNet(inputs)

    batch_loss = criterion(prediction,labels)
    total_loss_train += batch_loss.item()

    abs_value = torch.abs(prediction - labels)
    acc_train = (abs_value < tolerance).float().mean()

    total_acc_train.append(acc_train)

    batch_loss.backward()
    optimizer.step()
    optimizer.zero_grad()

  avg_loss_train = total_loss_train/len(train_dataloader)

  with torch.no_grad():
    for data in test_dataloader:
      inputs, labels = data

      prediction = BE_NeuralNet(inputs)

      batch_loss = criterion(prediction,labels)
      total_loss_test += batch_loss.item()

      abs_value = torch.abs(prediction - labels)
      acc_test = (abs_value < tolerance).float().mean()

      total_acc_test.append(acc_test)

    avg_loss_test = total_loss_test/len(test_dataloader)


  #Takes average accuracy of each epoch and converts to percent
  avg_acc_test = (sum(total_acc_test)/len(total_acc_test))*100
  avg_acc_train = (sum(total_acc_train)/len(total_acc_train))*100


  print(f'''Epoch: {epoch+1} Train_Loss: {avg_loss_train} Train_Accuracy: {avg_acc_train}
        Testing Loss: {avg_loss_test} Testing Accuracy: {avg_acc_test}''')
  print("=="*25)


Epoch: 1 Train_Loss: 0.019336148520603804 Train_Accuracy: 97.15589904785156
        Testing Loss: 0.02349986997997098 Testing Accuracy: 97.05056762695312
Epoch: 2 Train_Loss: 0.012529102013099312 Train_Accuracy: 97.85814666748047
        Testing Loss: 0.0200844901407303 Testing Accuracy: 96.06741333007812
Epoch: 3 Train_Loss: 0.026848877752146935 Train_Accuracy: 96.10253143310547
        Testing Loss: 0.005269431283686368 Testing Accuracy: 98.17416381835938
Epoch: 4 Train_Loss: 0.014799594807903233 Train_Accuracy: 97.64746856689453
        Testing Loss: 0.007171969549750082 Testing Accuracy: 97.05056762695312
Epoch: 5 Train_Loss: 0.009537398937714549 Train_Accuracy: 97.05056762695312
        Testing Loss: 0.042672226519438014 Testing Accuracy: 96.06741333007812
Epoch: 6 Train_Loss: 0.02907173943014457 Train_Accuracy: 96.27809143066406
        Testing Loss: 0.018134625569483646 Testing Accuracy: 95.08426666259766
Epoch: 7 Train_Loss: 0.024986792621562885 Train_Accuracy: 96.8398895263671